# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** First learned model for this lane, compared
against the Week-4 rule baseline on the exact same held-out rows and the exact same metrics.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'{len(df):,} rows, {df["client_id"].nunique()} clients')


30,000 rows, 32 clients


## 1. Method choice and why

**The question shape:** `is_declining_label` (`trend_direction == "down"`) is an observed
yes/no label, and the real use case is a ranked queue — "which pages first?" — so per the
`training-honest-models` toolkit this is a **classification problem evaluated by precision@K**,
not just accuracy or a bare AUC.

**Methods, in order of complexity:**
1. **Logistic Regression** — the readable starting point. If it can't beat the baseline, nothing
   fancier deserves the benefit of the doubt.
2. **Random Forest** — the "stronger" step up the toolkit's table. Only worth keeping if it earns
   its extra complexity over Logistic Regression on the same metric.
3. **A depth-3 Decision Tree** — added as a sanity check on the other end: if a tree three
   splits deep gets most of the way to Logistic Regression's score, that says the real signal in
   this lane is simple, and complexity isn't where the value is.

I'm not using clustering here — the lane's actual question is "score and rank," not "find
groups," so clustering doesn't fit the question shape.


In [2]:
# Backing check: confirm this is genuinely a binary framing (not close to all-one-class,
# which would make "prediction" trivial) before committing to classification.
print(df['trend_direction'].value_counts())
print()
print('is_declining base rate:', round((df['trend_direction']=='down').mean(), 3))


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining base rate: 0.542


## 2. Split design

**Grouped by `client_id`, not random row-level.** Content items from the same client share
systematic patterns — industry, template, editorial cadence, how aggressively that client's team
refreshes content. A random row-level split would let the model see some of a client's pages in
training and others in test, effectively memorizing client identity rather than learning a
transferable "this page is declining" signal. That's exactly what the data dictionary flags
`client_id` for: grouping and **client-holdout splits**, never as a feature.

80/20 split by client, fixed seed (`random_state=42`) for reproducibility.


In [3]:
from sklearn.model_selection import GroupShuffleSplit

NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct',
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]
# NOTE: trend_direction and trend_pct are excluded on purpose - they ARE the label (w03's
# leakage trap). impressions_last_30d / prev_30d are excluded too since they're the trend's
# raw inputs - using them would be the same leak one step removed.

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna('unknown').astype(str)

# log1p the heavy-tailed traffic counts, same as the prep script does
for c in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    df[f'log_{c}'] = np.log1p(df[c])
NUM_FINAL = [c for c in NUMERIC_FEATURES if c not in
             ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']] + \
            ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']

X = df[NUM_FINAL + CATEGORICAL_FEATURES]
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f'train rows: {len(train_idx):,}   test rows: {len(test_idx):,}')
print(f'client overlap between train and test: {len(overlap)} (must be 0)')
print(f'train base rate: {y.iloc[train_idx].mean():.3f}   test base rate: {y.iloc[test_idx].mean():.3f}')


train rows: 23,837   test rows: 6,163
client overlap between train and test: 0 (must be 0)
train base rate: 0.550   test base rate: 0.511


## 3. Train + compare vs my baseline

Same held-out test rows for everything below: the two learned models AND the Week-4 rule,
recomputed here so it's scored on the identical split rather than a different sample. Same
metrics too — precision@K at a few K values, ROC-AUC, and the base rate for reference (per the
`building-baselines` skill: precision@K means nothing without the base rate next to it).


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

pre = ColumnTransformer([
    ('num', StandardScaler(), NUM_FINAL),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

models = {
    'Logistic Regression': Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Random Forest':       Pipeline([('pre', pre), ('clf', RandomForestClassifier(
                                n_estimators=300, max_depth=8, min_samples_leaf=20,
                                random_state=42, n_jobs=-1))]),
    'Decision Tree (depth=3)': Pipeline([('pre', pre), ('clf', DecisionTreeClassifier(
                                max_depth=3, min_samples_leaf=200, random_state=42))]),
}

y_test = y.iloc[test_idx].values
proba = {}
for name, pipe in models.items():
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba[name] = pipe.predict_proba(X.iloc[test_idx])[:, 1]

# Week-4 baseline rule, recomputed here on the same rows (not reloaded from a possibly-stale CSV)
ctr_benchmark = {'top_3': 0.334128, 'page_1': 0.354760, 'striking': 0.255782,
                 'page_3_5': 0.142359, 'deep': 0.055415}
stale = (df['days_since_last_update'] >= 90).astype(int)
visible = (df['impressions_90d'] >= 100).astype(int)
position_ok = df['position_tier'].isin(['top_3', 'page_1', 'striking', 'page_3_5']).astype(int)
benchmark = df['position_tier'].map(ctr_benchmark).fillna(0)
ctr_gap = (benchmark - df['ctr']).clip(lower=0)
baseline_score = (stale * visible * position_ok * ctr_gap * np.log1p(df['impressions_90d'])).values
proba['Baseline rule (Week 4)'] = baseline_score[test_idx]


In [5]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order[:k]].mean())

Ks = [20, 50, 100, 200]
rows = []
for name, scores in proba.items():
    row = {'method': name, 'ROC-AUC': round(float(roc_auc_score(y_test, scores)), 3)}
    for k in Ks:
        row[f'precision@{k}'] = round(precision_at_k(y_test, scores, k), 3)
    rows.append(row)

comparison = pd.DataFrame(rows).set_index('method')
comparison.loc['base rate (reference)'] = [np.nan] + [round(float(y_test.mean()), 3)] * len(Ks)
comparison


,ROC-AUC,precision@20,precision@50,precision@100,precision@200
method,,,,,
Logistic Regression,0.616,0.700,0.720,0.700,0.710
Random Forest,0.605,0.600,0.560,0.570,0.550
Decision Tree (depth=3),0.581,0.600,0.540,0.590,0.580
Baseline rule (Week 4),0.506,0.550,0.660,0.650,0.655
base rate (reference),NaN,0.511,0.511,0.511,0.511


**Reading the table:** Logistic Regression wins on every metric here — ROC-AUC and all four
precision@K values — and Random Forest, despite being the "stronger" model in the toolkit's
table, does not beat it. That's the honest result, not a tuning failure to fix: with ~26 features
and a fairly linear-looking relationship between traffic/age and decline, the extra flexibility
of a forest isn't earning its complexity here. The depth-3 tree gets close to Logistic Regression
despite being readable in one glance — a good sign the real signal in this lane is simple.

The **baseline rule's ROC-AUC sits near 0.50** (barely better than chance) while its
precision@K is still meaningfully above the base rate. That's not a contradiction: the baseline
was never built to *rank everything* by decline probability — it was built to surface a specific,
narrow opportunity (visible + stale + underperforming CTR for its position). It's answering a
related but different question than "predict decline," and Logistic Regression, trained
explicitly on the label, naturally does better at that exact task. Worth saying plainly rather
than reading it as a clean win: **Logistic Regression beats the baseline at the thing it was
trained to do; it doesn't automatically mean the baseline was a bad rule for what it does.**


## 4. Errors and interpretation

Permutation importance on the winning model (Logistic Regression), then concrete false positive
and false negative cases from the test set.


In [6]:
from sklearn.inspection import permutation_importance

best_pipe = models['Logistic Regression']
result = permutation_importance(best_pipe, X.iloc[test_idx], y_test,
                                 n_repeats=8, random_state=42, scoring='roc_auc', n_jobs=-1)
feat_names = NUM_FINAL + CATEGORICAL_FEATURES
importance = pd.Series(result.importances_mean, index=feat_names).sort_values(ascending=False)
print('Top 8 permutation importances (AUC drop when shuffled):')
print(importance.head(8).round(4))


Top 8 permutation importances (AUC drop when shuffled):
log_impressions_90d    0.0803
log_clicks_90d         0.0575
content_age_days       0.0343
impression_tier        0.0339
avg_position           0.0257
log_sessions_90d       0.0235
word_count             0.0092
days_with_sessions     0.0071
dtype: float64


**Top features, sanity-checked:** `log_impressions_90d`, `log_clicks_90d`, `content_age_days`,
`impression_tier`, and `avg_position` lead. All plausible, none suspiciously perfect — traffic
volume and content age are exactly the kind of thing that would relate to whether a page's
30-day trend is heading down, and nothing here even resembles `trend_direction`/`trend_pct`
sneaking back in through a side door. That's the leakage sanity check the toolkit asks for:
a clean, boring top-feature list is a good sign, not a disappointing one.


In [7]:
test_df = df.iloc[test_idx].copy()
test_df['pred_proba'] = proba['Logistic Regression']
test_df['actual'] = y_test

cols = ['content_id', 'impressions_90d', 'avg_position', 'content_age_days',
        'days_since_last_update', 'ctr', 'pred_proba', 'actual']

false_positives = test_df[(test_df['pred_proba'] > 0.7) & (test_df['actual'] == 0)].sort_values('pred_proba', ascending=False)
false_negatives = test_df[(test_df['pred_proba'] < 0.3) & (test_df['actual'] == 1)].sort_values('pred_proba')

print(f'false positives (pred>0.7, actually stable): {len(false_positives)}')
print(f'false negatives (pred<0.3, actually declining): {len(false_negatives)}')
print()
print('3 false positives:')
print(false_positives[cols].head(3).to_string(index=False))
print()
print('3 false negatives:')
print(false_negatives[cols].head(3).to_string(index=False))


false positives (pred>0.7, actually stable): 570
false negatives (pred<0.3, actually declining): 191

3 false positives:
          content_id  impressions_90d  avg_position  content_age_days  days_since_last_update  ctr  pred_proba  actual
content_7be5f150dc65              290           5.9                96                      20  0.0    0.954100       0
content_41baf0722ad9             3115          12.8               275                     104  0.0    0.942681       0
content_5d5653c4eb4f            15101           5.7               421                       7  0.0    0.932321       0

3 false negatives:
          content_id  impressions_90d  avg_position  content_age_days  days_since_last_update  ctr  pred_proba  actual
content_d1e915d03c28                2          45.0               537                     104  0.0    0.063800       1
content_e18144cbd19d                3           2.0               545                      20  0.0    0.066531       1
content_c268b1716236      

**Why these are hard:**

- **False positives** are large, established pages (thousands to tens of thousands of
  impressions, months old) sitting at **0.00% CTR** — the model reads "big page, zero clicks" as
  a strong decline signal, but `trend_direction` measures whether *impression volume* is
  falling, not whether CTR is low. A page can hold flat impressions with a permanently bad CTR;
  that's a real problem, just not the one this label describes. The model is picking up a
  correlated-but-different signal.
- **False negatives** are the mirror image: tiny pages (2-3 impressions in 90 days), some very
  old, one even sitting at position 2. At that volume, `trend_pct` (last-30d vs prev-30d
  impressions) swings wildly on a difference of one or two impressions — exactly the small-
  denominator noise the data dictionary warns about for `trend_pct`. The model's log1p transform
  compresses these tiny-volume pages together with thousands of other quiet-but-stable pages, so
  it has almost nothing to distinguish "about to flip down" from "always this quiet."

Both error types point at the same root cause: `trend_direction`'s ±20% threshold behaves
differently at large volume (stable, meaningful) than at tiny volume (noisy, mechanical) — a
limitation of the label itself, not something more features would fix.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
